# Text Analytics Lab 5: Information Extraction

This lab looks at the tasks of named entity recognition (NER) and relation extraction, and walks you through three key approaches: CRFs and transformers for named entity recognition, formulated as a kind of sequence labelling or token classification task; and LLMs for relation extraction, formulated as a kind of question answering task.

### Outcomes
After completing the lab, you will be able to...
* Apply CRFs to NER.
* Apply pretrained transformers to NER using a token classification setup.
* Compute performance metrics for NER.
* Formulate relation extraction as a question answering prompt for an LLM.

# 1. Named Entity Recognition (NER) with CRF

Named entity recognition is the task of identifying mentions of entities in text, such as people, locations, organisations and times. It is iften modelled as a sequence labelling task, where the goal is to label each token as either part of an entity of a specific type, or as not part of a named entity. Many entities span more than one token, such as the person "Ada Lovelace" or the organisation "University of Bristol". To show where these spans start and end, in addition to tagging the entity type (such as person or organisation), we also tag each token as 'beginning' (at the start of an entity span), 'inside' (continuing an entity span), or 'outside' (not part of an entity). Hence, the complete set of tags include "O" for 'outside', along with "B-Person", "I-Person", "B-Organisation", "I-Organisation", etc., for each entity type. 

Let's load some NER data consisting of English news articles.

In [ ]:
from datasets import load_dataset

cache_dir = "./data_cache"

# The data is already divided into training and test sets.
# Load the training set:
train_dataset = load_dataset(
    "conll2003",
    split="train",
    cache_dir=cache_dir,
)
print(f"Training dataset with {len(train_dataset)} instances loaded")

In [ ]:
# Load the test set:
val_dataset = load_dataset(
    "conll2003",
    split="validation",
    cache_dir=cache_dir,
)
print(f"Validation dataset with {len(val_dataset)} instances loaded")

In [ ]:
# Load the test set:
test_dataset = load_dataset(
    "conll2003",
    split="test",
    cache_dir=cache_dir,
)
print(f"Test dataset with {len(test_dataset)} instances loaded")

Let's take a look at one of the instances in the training set:

In [ ]:
train_dataset[0]

The text is already tokenised -- this is because the NER tags are aligned with the tokens. 
As well as NER tags, the dataset includes PoS tags and Chunk tags, which provide grammatical information about each token.

### Part of Speech (POS) Tags

POS tags are useful features for NER with CRFs. They encode the syntactic category of a word -- types of nouns, verbs, adjectives, and so on. This syntactic information about each word in a sentence can help the model to recognise phrases referring to named entities, e.g., proper nouns are typically names of entities. POS tags of neighbouring words can also help to identify named entity spans, as a name if likely to occur in a particular place in a sentence, such as when it is the object of a verb. The task of POS tagging is more general and, for many languages, easier to learn than NER itself. Therefore, a POS tagger can be pretrained and applied to many different NER tasks, as well as other NLP tasks. 

The tags are all stored by their indexes, rather than as the tags themselves. The mapping of the PoS tags to their indexes is:

```
{'"': 0, "''": 1, '#': 2, '$': 3, '(': 4, ')': 5, ',': 6, '.': 7, ':': 8, '``': 9, 'CC': 10, 'CD': 11, 'DT': 12,
 'EX': 13, 'FW': 14, 'IN': 15, 'JJ': 16, 'JJR': 17, 'JJS': 18, 'LS': 19, 'MD': 20, 'NN': 21, 'NNP': 22, 'NNPS': 23,
 'NNS': 24, 'NN|SYM': 25, 'PDT': 26, 'POS': 27, 'PRP': 28, 'PRP$': 29, 'RB': 30, 'RBR': 31, 'RBS': 32, 'RP': 33,
 'SYM': 34, 'TO': 35, 'UH': 36, 'VB': 37, 'VBD': 38, 'VBG': 39, 'VBN': 40, 'VBP': 41, 'VBZ': 42, 'WDT': 43,
 'WP': 44, 'WP$': 45, 'WRB': 46}
 ```
You can find out what the tags mean at https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html. 

### Chunk Tags

Chunk tags are another kind of grammatical information about a sentence. Chunking splits a sentence into phrases, e.g., a noun phrase "the red car", which consists of tokens that relate to a single entity. The chunk tags indicate what kind of phrase each token belongs to. The tag set is listed below -- you don't need to understand what all of these mean, but if you are interested, you can read more in the NLTK book: https://www.nltk.org/book/ch07.html .
```
chunk_tags = {'O': 0, 'B-ADJP': 1, 'I-ADJP': 2, 'B-ADVP': 3, 'I-ADVP': 4, 'B-CONJP': 5, 'I-CONJP': 6, 'B-INTJ': 7, 'I-INTJ': 8, 'B-LST': 9, 'I-LST': 10, 'B-NP': 11, 'I-NP': 12, 'B-PP': 13, 'I-PP': 14, 'B-PRT': 15, 'I-PRT': 16, 'B-SBAR': 17, 'I-SBAR': 18, 'B-UCP': 19, 'I-UCP': 20, 'B-VP': 21, 'I-VP': 22}  
```

### NER Tag IDs

The mapping from indexes to NER tags to indexes is:

```
{'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8}
```

The BIO tagging scheme here is a little different to the standard BIO approach discussed in the lecture: the B tags are only used if an entity span immediately follows another and the B tag is needed to indicate the start. Otherwise, entity tokens are tagged with "I-". 

In [ ]:
ner_tag_mapping = {0: 'O', 1:'B-PER', 2:'I-PER', 3:'B-ORG', 4:'I-ORG', 5:'B-LOC', 6:'I-LOC', 7:'B-MISC', 8:'I-MISC'}

Let's put the NER data in the right format for NLTK's CRFTagger class. The CRFTagger requires that the data is a sequence of tuples, where each tuple contains the input token and the output tag. 

In [ ]:
train_set = [list(zip(s['tokens'], [ner_tag_mapping[tok] for tok in s['ner_tags']])) for s in train_dataset][:-1]
test_set = [list(zip(s['tokens'], [ner_tag_mapping[tok] for tok in s['ner_tags']])) for s in test_dataset][:-1]
test_tokens = [s['tokens'] for s in test_dataset][:-1]
test_tags = [[ner_tag_mapping[tok] for tok in s['ner_tags']] for s in test_dataset][:-1]

Now, let's train a CRF tagger on our training set. The method you need to use from NLTK is the [train method of the conditional random field (CRF)](https://www.nltk.org/_modules/nltk/tag/crf.html). You need to call the constructor with default arguments, then the train() function.

**TO-DO 1:** Write a function to train and return a CRF named entity recogniser.

NOTE: for this step, you need to have the package pycrfsuite installed. If you encounter an error saying that this module cannot be found, run ``conda install python-crfsuite``. 

In [ ]:
import nltk

# Train a CRF NER tagger
def train_CRF_NER_tagger(train_set):
    ### WRITE YOUR OWN CODE HERE
    tagger = nltk.tag.CRFTagger()
    tagger.train(train_set, 'model.crf.tagger')
    return tagger  # return the trained model

tagger = train_CRF_NER_tagger(train_set)

Get some predictions from the tagger:

In [ ]:
predicted_tags = tagger.tag_sents(test_tokens)

Let's see how well the tagger is performing. In NER, we usually evaluate performance by finding **correctly matched entities, rather than correctly tagged tokens**. Only an exact entity match counts as correct. This is quite a strict metric, but imagine if your tagger identifies only "York" as a location entity in a piece of text that mentions "New York" -- returning half of the span is misleading, as "York" refers to a different city. Often, the whole entity span is needed to recognise it correctly. Therefore, to compute precision, recall and F1 score, we need to compute true positives, false positives and false negatives by looking for the predicted entity spans and the gold-labelled entity spans in the test set.

The code below contains a function that extract a list of spans from the tagged sentences. The next function calls extract_spans() and computes the precision, recall and f1 scores. However, the function is incomplete.

Run the cal_span_level_F1() function below to compute span-level F1 scores for the predictions. Have a look at the results. Which types of entity are being recognised well and which are very poor?

In [ ]:
import numpy as np

def extract_spans(tagged_sents):
    """
    Extract a list of tagged spans for each named entity type, 
    where each span is represented by a tuple containing the 
    start token and end token indexes.
    
    returns: a dictionary containing a list of spans for each entity type.
    """
    spans = {}
        
    for sidx, sent in enumerate(tagged_sents):
        start = -1
        entity_type = None
        for i, (tok, lab) in enumerate(sent):
            if 'B-' in lab:
                start = i
                end = i + 1
                entity_type = lab[2:]
            elif 'I-' in lab:
                end = i + 1
            elif lab == 'O' and start >= 0:
                
                if entity_type not in spans:
                    spans[entity_type] = []
                
                spans[entity_type].append((start, end, sidx))
                start = -1      
        # Sometimes an I-token is the last token in the sentence, so we still have to add the span to the list
        if start >= 0:    
            if entity_type not in spans:
                spans[entity_type] = []
                
            spans[entity_type].append((start, end, sidx))
                
    return spans


def cal_span_level_f1(test_sents, test_sents_with_pred):
    # get a list of spans from the test set labels
    gold_spans = extract_spans(test_sents)

    # get a list of spans predicted by our tagger
    pred_spans = extract_spans(test_sents_with_pred)
    
    # compute the metrics for each class:
    f1_per_class = []
    
    ne_types = gold_spans.keys()  # get the list of named entity types (not the tags)
    
    for ne_type in ne_types:
        # compute the confusion matrix
        true_pos = 0
        false_pos = 0
        
        for span in pred_spans[ne_type]:
            if span in gold_spans[ne_type]:
                true_pos += 1
            else:
                false_pos += 1
                
        false_neg = 0
        for span in gold_spans[ne_type]:
            if span not in pred_spans[ne_type]:
                false_neg += 1
                
        if true_pos + false_pos == 0:
            precision = 0
        else:
            precision = true_pos / float(true_pos + false_pos)
            
        if true_pos + false_neg == 0:
            recall = 0
        else:
            recall = true_pos / float(true_pos + false_neg)
        
        if precision + recall == 0:
            f1 = 0
        else:
            f1 = 2 * precision * recall / (precision + recall)
            
        f1_per_class.append(f1)
        print(f'F1 score for class {ne_type} = {f1}')
        
    print(f'Macro-average f1 score = {np.mean(f1_per_class)}')

cal_span_level_f1(test_set, predicted_tags)

We can try to help the CRF tagger by adding some more features -- the great thing about the CRF model is that we can easily add in any features that we think may be helpful, and these do not need to be conditionally independent of one another.

In the CRFTagger class, the ```_get_features()``` method extracts the features from the tokens. We can overwrite this method with our own version that provides additional features.

**TO-DO 2:** Add in the previous and next works as features. Be careful with the start and end of the sequence where there is no previous or next word.

In [ ]:
import re, unicodedata

class CustomCRFTagger(nltk.tag.CRFTagger):
    _current_tokens = None
    
    def _get_features(self, tokens, idx):
            """
            Extract basic features about this word including
                - Current word
                - is it capitalized?
                - Does it have punctuation?
                - Does it have a number?
                - Suffixes up to length 3

            Note that : we might include feature over previous word, next word etc.

            :return: a list which contains the features
            :rtype: list(str)
            """
            token = tokens[idx]

            feature_list = []

            if not token:
                return feature_list

            # Capitalization
            if token[0].isupper():
                feature_list.append("CAPITALIZATION")

            # Number
            if re.search(self._pattern, token) is not None:
                feature_list.append("HAS_NUM")

            # Punctuation
            punc_cat = {"Pc", "Pd", "Ps", "Pe", "Pi", "Pf", "Po"}
            if all(unicodedata.category(x) in punc_cat for x in token):
                feature_list.append("PUNCTUATION")

            # Suffix up to length 3
            if len(token) > 1:
                feature_list.append("SUF_" + token[-1:])
            if len(token) > 2:
                feature_list.append("SUF_" + token[-2:])
            if len(token) > 3:
                feature_list.append("SUF_" + token[-3:])

                
            # Current word
            feature_list.append("WORD_" + token)
            
            ### WRITE YOUR OWN CODE HERE ###
            
            

            
                
            ####

            return feature_list
                

**TO-DO 3:** Train your custom CRF tagger with your ``_get_features`` method, then test it below. How does it compare to the default tagger? Why do you think adding the new features change the performance in this way? 

The results show how important it is understand your choice of features.

In [ ]:
# Train a CRF NER tagger
def train_CustomCRF_NER_tagger(train_set):
    ### WRITE YOUR OWN CODE HERE
    tagger = 
    
    
    return tagger  # return the trained model

tagger = train_CustomCRF_NER_tagger(train_set)

In [ ]:
predicted_tags = tagger.tag_sents(test_tokens)
cal_span_level_f1(test_set, predicted_tags)

Part-of-speech tags often provide useful information for identifying entities, e.g., by identifying nouns that may be part of a name. Let's see if they help when dealing with the CoNLL 2003 dataset...

We can use the pretrained PoS tagger from NLTK to tag a sentence, as shown in the following cell:

In [ ]:
# Download the packages for English PoS tagging
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

# Some arbitrary token sequence to see how it works...
example_sentence = ["PoS", "tags", "often", "provide", "useful", "information", "for", "identifying", "entities"]

# Tag the sentence...
pos_tagged_tokens = nltk.pos_tag(example_sentence)

In [ ]:
pos_tagged_tokens  # look at the results

**TO-DO 4:** Complete the code below to define another custom CRF tagger that also include PoS tags as features. Then, run the final cells to train and test the CRF tagger with PoS tags. How do the results compare with the previous versions?

In [ ]:
# *** Improve the CRF NER tagger using parts of speech (see lab 5) as additional features.
class CRFTaggerWithPOS(CustomCRFTagger):
    _current_tokens = None
    
    def _get_features(self, tokens, index):
        """
        Extract the features for a token and append the POS tag as an additional feature.
        """
        basic_features = super()._get_features(tokens, index)
        
        # Get the pos tags for the current sentence and save it
        if tokens != self._current_tokens:
            self._pos_tagged_tokens = nltk.pos_tag(tokens)
            self._current_tokens = tokens
            
            
        ### WRITE YOUR OWN CODE HERE

        ###
        
        return basic_features

In [ ]:
# Train a CRF NER tagger
def train_CRF_NER_tagger_with_POS(train_set):
    ### WRITE YOUR OWN CODE HERE
    tagger = 
    
    return tagger  # return the trained model

tagger = train_CRF_NER_tagger_with_POS(train_set)

In [ ]:
predicted_tags = tagger.tag_sents(test_tokens)
cal_span_level_f1(test_set, predicted_tags)

**TO-DO 5:** Try to further improve the CRF NER tagger by adding more features. For example, could your model benefit from Chunk tags or some other features?

# 2. NER with Transformers

The CRF depends on careful feature design to provide a suitable representation of each token and its context for tagging a sequence. An alternative approach is to use a pretrained transformer to provide a sequence of contextual embeddings of tokens. These embeddings capture information about the meaning of each token in a particular text sequence, so they incorporate context information that can be used to classify the tokens with NER tags. 

A key issue is that each pretrained model may use a different tokenizer. Since the tokenizer determines the vocabulary -- the set of tokens seen during training -- they will not work well if we split tokens using a different tokenizer. However, our dataset is pre-tokenized, and it was not split up in the same way that a model such as BERT would tokenize the text.

What can we do to fix this? We can use the tokenizer for our chosen model, along with the 'is_split_into_words' argument, and implement a function that applies the correct tokenizer and puts the corresponding labels into the sequence. 

In [ ]:
def tokenize_and_align_labels(dataset):
    tokens, tags = dataset["tokens"], dataset["ner_tags"]

    # The text is already split into tokens. We can tell this to the tokenizer by setting is_split_into_words=True.
    # If the pre-split token is not recognised by the tokenizer, it will be split into multiple sub-tokens. In this case, we need to align the labels with the sub-tokens.
    tokenized_inputs = tokenizer(tokens, is_split_into_words=True)
    dataset["input_ids"] = tokenized_inputs["input_ids"]
    dataset["attention_mask"] = tokenized_inputs["attention_mask"]

    sub_token_labels = []
    
    # Loop over each of the examples in the dataset.
    for i, labels in enumerate(tags):

        # For each token in this example, if the original token was not recognised by the tokenizer, 
        # it will be split into multiple sub-tokens. 
        # word_ids will tell us the index of the original token that each sub-word token is a part of:
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        label_ids = []
        
        # Loop over the sub-tokens corresponding to this current original token at position i in the original sequence:
        previous_word_idx = None
        for j, word_idx in enumerate(word_ids):
            # Special tokens have a word id that is None. We set the label to -100 so they are automatically
            # ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
        
            # We apply the label for this original token to its first subtoken. 
            # So, if this subtoken has a different index to the last one, it must be the first part of a new original token. 
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])

            # Subsequent subtokens get -100, which causes them to be ignored in the loss function. 
            # This prevents long words from dominating the loss function and simplifies alignment when making predictions. 
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        sub_token_labels.append(label_ids)

    # Create a new field in the dataset containing the aligned labels:
    dataset["labels"] = sub_token_labels

    return dataset

**TO-DO 6:** Create a tokenizer object for the model "huawei-noah/TinyBERT_General_4L_312D" (as we did in lab 4). You need to pass an argument ```do_lower_case=True```  because this tokenizer does not recognise capitalised words, so we need to lowercase the input text.

In [ ]:
from transformers import AutoTokenizer

### WRITE YOUR OWN CODE HERE ###
tokenizer = 

**TO-DO 7:** Use the tokenize_and_align_labels function to preprocess the CONLL dataset (`train_dataset`, `val_dataset`, and `test_dataset` objects) and prepare it for training a transformer-based NER model. Hint: you can use the map method from the Datasets library with the argument ```batched=True```. 

In [ ]:
### WRITE YOUR OWN CODE HERE ###


When we did document classification in lab 4, we used a SequenceClassification class to create a transformer model with a sequence classifier 'head' layer on top. Now, we need to use a TokenClassification class instead, which will put a token classifier head on top of the pretrained transformer. The token classifier head maps each token's contextualised embedding to probabilities over NER tags.

**TO-DO 8:** Complete the code below to create the model object.

In [ ]:
from transformers import AutoModelForTokenClassification

number_of_labels = len(ner_tag_mapping)

### COMPLETE THE CODE HERE ###
model = 


For speed, let's freeze all the weights in the pretrained transformer part of the model (you can try unfreezing them later):

In [ ]:
for param in model.bert.parameters():
    param.requires_grad = False

Now we can train the token classification model. The rest of the setup is very similar to the SequenceClassification setup from the previous lab:

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="transformer_checkpoints",  # specify the directory where models weights will be saved a certain points during training (checkpoints)
    num_train_epochs=3, # A sensible and sufficient number to use for the to-dos below
    per_device_train_batch_size=16,  # you can decrease this if memory usage is too high while training
    logging_steps=50,  # how often to print progress during training
)

In [ ]:
from transformers import Trainer
from torch import nn
from transformers import DataCollatorForTokenClassification

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tok_train_dataset,  # Check the names of these variables in your code above and correct them if necessary. 
    eval_dataset=tok_val_dataset,
    data_collator=DataCollatorForTokenClassification(tokenizer)  # this is needed to handle the padding of different length input sequences inside each batch. 
)

In [ ]:
# train...
trainer.train()

Now, we can obtain predictions from the trained model on our test dataset using the predict() method of the trainer.

In [ ]:
predictions, labels, metrics = trainer.predict(tok_test_dataset)

# get the most likely labels for each token from the probabilities provided by the previous line. 
pred_label_ids = np.argmax(predictions, axis=2)

We need to skip tokens with a training label of -100, as these are parts of a word or special tokens. 

In [ ]:
def remove_ignored_tokens(pred_label_ids, true_labels):
    pred_labels = [[ner_tag_mapping[p] for (p, l) in zip(sentence_predictions, sentence_labels) if  l != -100] for (sentence_predictions, sentence_labels) in zip(pred_label_ids, true_labels)]
    true_labels = [[ner_tag_mapping[l] for l in sentence_labels if  l != -100] for sentence_labels in true_labels]

    return pred_labels, true_labels

true_labels = tok_test_dataset["labels"]
pred_labels, true_labels = remove_ignored_tokens(pred_label_ids, true_labels)

Now, we can evaluate the results. As an alternative to the span-level F1 score function above, we can use the seqeval library, as in the next cell. Performance will be very weak because the TinyBERT weights were frozen and we did not train the model very long, so it has limited capacity to learn in this case. 

**TO-DO 9:** How could you improve the TinyBERT model's performance? 

In [ ]:
from seqeval.metrics import classification_report

print(classification_report(true_labels, pred_labels))

# Relation Extraction with LLMs

For this part, we will look at how to prompt a language model to extract relations from a piece of text. Performance will not be great, as we will use a small Flan-T5 model to demonstrate the idea -- stronger LLMs will probably give more reliable results.

First, let's load up that generative LM model:

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("andresnowak/Qwen3-0.6B-instruction-finetuned")
model = AutoModelForCausalLM.from_pretrained("andresnowak/Qwen3-0.6B-instruction-finetuned")

We can define a function for prompting the pretrained model to generate some new text. Hopefully, the new text will be the relation between the subject and object entities, if the model can follow our instructions.

Notice that the prompt contains not just instructions, but also a few examples (for few-shot in-context learning).

In [ ]:
def extract_relation(model, tokenizer, sentence, subject, object_):
    prompt = f"""
    Extract relations of type Located_In, Work_For, OrgBased_In, Live_In, Kill, or No_Relation. 
    Example 1: Sentence = 'The director of BigCorp is Alison Potatosson.', Subject = 'Alison Potatosson', Object = 'BigCorp', Relation = 'Work_For'."
    Example 2: Sentence = 'London is busy today. Mary walks in the park.', Subject = 'London', Object = 'Mary', Relation = 'No_Relation'."
    Example 3: Sentence = 'Bristol nightclub Motion has moved to a new location.', Subject = 'Motion', Object = 'Bristol', Relation = 'OrgBased_In'."
    Example 4: Sentence = {sentence}, Subject = {subject}, Object = {object_}, Relation = """

    inputs = tokenizer(prompt, return_tensors="pt")
    output_ids = model.generate(**inputs, max_new_tokens=3)
    return tokenizer.decode(output_ids[0][-3:], skip_special_tokens=True)

Let's make a test sentence to try it out:

In [ ]:
sentence = "Satya Nadella is the Chairman and CEO of Microsoft, a position he has held since taking over from Steve Ballmer in 2014. "
subject = "Satya Nadella"
object_ = "Microsoft"
pred = extract_relation(model, tokenizer, sentence, subject, object_)
print(f"Sentence: {sentence}")
print(f"Predicted relation: {pred}")


We could change the prompt to use more of a question-answering style, e.g. "What is the relation between [subject] and [object] in the sentence [sentence]?", as shown below:

In [ ]:
def extract_relation(model, tokenizer, sentence, subject, object_):
    prompt = f"""
    Extract relations of type Located_In, Work_For, OrgBased_In, Live_In, Kill, or No_Relation. 
    Example 1: Sentence = 'The director of BigCorp is Alison Potatosson.' How is 'Alison Potatosson' related to 'BigCorp'? Relation = 'Work_For'."
    Example 2: Sentence = 'London is busy today. Mary walks in the park.', How is 'London' related to 'Mary'? Relation = 'No_Relation'."
    Example 3: Sentence = 'Bristol nightclub Motion has moved to a new location.', How is 'Motion' related to 'Bristol'? Relation = 'OrgBased_In'."
    Example 4: Sentence = {sentence} How is {subject} related to {object_}? Relation = """

    inputs = tokenizer(prompt, return_tensors="pt")
    output_ids = model.generate(**inputs, max_new_tokens=3)
    return tokenizer.decode(output_ids[0][-3:], skip_special_tokens=True)

In [ ]:
pred = extract_relation(model, tokenizer, sentence, subject, object_)
print(f"Sentence: {sentence}")
print(f"Predicted relation: {pred}")

To properly test this, we could load a dataset to demonstrate the approach. The code below tries out one example -- it doesn't do the job very well because this example is harder. It may need fine-tuning or more careful propmting. 

**OPTIONAL TO-DO 10:** Try to improve the LM's performance on this task, or test it on simpler IE tasks. 

In [ ]:
from datasets import load_dataset

dataset = load_dataset("DFKI-SLT/conll04")

In [ ]:
def retrieve_entity_from_indexes(tokens, start, end):
    return " ".join(tokens[start:end])

In [ ]:
# Demo
example = dataset["train"][0]  # just look at the first example for demonstration purposes
print(f"Sentence: {" ".join(example['tokens'])}")

for relation in example["relations"]:

    i = relation['head']
    j = relation['tail']
    true_rel = relation['type']

    subject = retrieve_entity_from_indexes(example["tokens"], example["entities"][i]['start'], example["entities"][i]['end'])
    object_ = retrieve_entity_from_indexes(example["tokens"], example["entities"][j]['start'], example["entities"][j]['end'])
    pred = extract_relation(model, tokenizer, " ".join(example["tokens"]), subject, object_)
    print(f"Subject entity: {subject}")
    print(f"Object entity: {object_}")
    print(f"Predicted relation: {pred}")


    print(f"True relation: {true_rel}\n")